<a href="https://colab.research.google.com/github/IsuPaul/Indpendent-Research/blob/IR_Codes/IR_Source_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENHANCED MACHINE LEARNING MODEL FOR 3D PRINTING CONCRETE COMPRESSIVE STRENGTH PREDICTION

## Imports

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xbt
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import shap
import io
from sklearn.utils import resample
from sklearn.preprocessing import scale
from sklearn.tree import DecisionTreeRegressor
from sklearn import svm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.model_selection import learning_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.inspection import PartialDependenceDisplay
from google.colab import files

## DATA MANAGEMENT AND CLEANING

### Visualization and Description of Uncleaned Data

#### Database Import

In [ ]:
df = pd.read_csv('Datasets.csv', encoding='latin-1')

In [ ]:
#df.head()

In [ ]:
#df.describe()

### Determining the type of data in each column

In [ ]:
#df.dtypes

### Converting all data to float

In [ ]:
cols_to_convert = ['Carbonated RFA (Kg/m³)', 'PP (Kg/m³)', 'Silica Powder (Kg/m³)', 'AT (Kg/m³)']

for col in cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

# Displaying types to confirm
#print(df.dtypes)

### Checking for missing data

In [ ]:
#df.isnull().sum()

### Fixing Missing Data

In [ ]:
imputer_gbt = IterativeImputer(max_iter=10, random_state=42)
imputed_data=imputer_gbt.fit_transform(df)
imputed_data = np.maximum(imputed_data, 0)
imputed_df=pd.DataFrame(imputed_data,columns=df.columns)

### Reconfirm to check for missing data

In [ ]:
#imputed_df.isnull().sum()

### Features Distributions

In [ ]:
for i in imputed_df.columns:
    plt.figure(figsize=(2.5, 2.5))
    plt.hist(imputed_df[i])
    plt.xlabel(f"{i}",fontsize=7)
    plt.ylabel("Frequency",fontsize=7)
    plt.title(" ")
    plt.tight_layout()
    plt.savefig(f"{i.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')}.jpg")
    plt.close('all')

## MODEL DEVELOPEMENTS

### Variable Assignments

#### Assigning Variables for XGBOOST, RF, DT, and MLR

In [ ]:
X=imputed_df.drop('Compressive Strength (MPa)',axis=1).copy()
y=imputed_df['Compressive Strength (MPa)'].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=42)

#### Scaling the dataframe and assigning variables for SVM and KNN

In [ ]:
target_col = 'Compressive Strength (MPa)'
df_min = imputed_df.min()
df_max = imputed_df.max()
df_range = df_max - df_min
# Scale X_train and X_test using global stats
Xs_train = (X_train - df_min.drop(target_col)) / df_range.drop(target_col)
Xs_test = (X_test - df_min.drop(target_col)) / df_range.drop(target_col)
# Scale y_train and y_test using global stats
ys_train = (y_train - df_min[target_col]) / df_range[target_col]
ys_test = (y_test - df_min[target_col]) / df_range[target_col]

#### Assigning Variables for Material and Process Parameter Models

In [ ]:
target_col = 'Compressive Strength (MPa)'
# 1. Material Model Subset
columns_to_remove = ['Nozzle Speed (mm/s)', 'DIA (mm)', 'Layer Depth (mm)', 'Layer Width (mm)']
X_material = imputed_df.drop(columns_to_remove + [target_col], axis=1).copy()
y_material = imputed_df[target_col].copy()
X_train_mat, X_test_mat, y_train_mat, y_test_mat = train_test_split(X_material, y_material, test_size=0.3, random_state=42)

# 2. Printing Model Subset
X_printing = imputed_df[['Nozzle Speed (mm/s)', 'DIA (mm)', 'Layer Depth (mm)', 'Layer Width (mm)']].copy()
y_printing = imputed_df[target_col].copy()
X_train_print, X_test_print, y_train_print, y_test_print = train_test_split(X_printing, y_printing, test_size=0.3, random_state=42)

print('Data splits for Material and Printing models re-initialized.')

### Hyperparameter Tunning of Models

In [ ]:
param_grids = {
    'XGBoost': {'learning_rate': [0.1, 0.2], 'max_depth': [1,3,4,], 'n_estimators': [500, 1000]},
    'SVM': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'], 'kernel': ['rbf']},
    'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20, None]},
    'DecisionTree': {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]},
    'KNN': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}
}

####### Models to optimize
models_to_tune = {
    'XGBoost': xbt.XGBRegressor(random_state=42),
    'SVM': svm.SVR(),
    'RandomForest': RandomForestRegressor(random_state=42),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'KNN': KNeighborsRegressor()
}

optimized_models = {}

####### Optimization and Evaluation Loop
for name, model in models_to_tune.items():
    print(f'Optimizing {name}...')
    is_scaled = name in ['SVM', 'KNN']
    cur_X_tr, cur_y_tr = (Xs_train, ys_train) if is_scaled else (X_train, y_train)
    cur_X_ts, cur_y_ts = (Xs_test, ys_test) if is_scaled else (X_test, y_test)

    grid = GridSearchCV(model, param_grids[name], cv=10, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(cur_X_tr, cur_y_tr)
    best_model_obj = grid.best_estimator_
    optimized_models[name] = best_model_obj

    print(f'Best Params for {name}: {grid.best_params_}')

    for d_name, x_set, y_set in [('Train', cur_X_tr, cur_y_tr), ('Test', cur_X_ts, cur_y_ts)]:
        preds = best_model_obj.predict(x_set)
        if is_scaled:
            y_actual = y_set * df_range[target_col] + df_min[target_col]
            p_actual = preds * df_range[target_col] + df_min[target_col]
        else:
            y_actual, p_actual = y_set, preds

        print(f'{name} {d_name} -> MAE: {mean_absolute_error(y_actual, p_actual):.4f}, RMSE: {np.sqrt(mean_squared_error(y_actual, p_actual)):.4f}, R2: {r2_score(y_actual, p_actual):.4f}')

### Training Models

#### XGBOOST

##### Training the model

In [ ]:
# Training
clf=xbt.XGBRegressor()
clf.fit(X_train, y_train)
# Evaluation
for name, x_eval, y_eval in [('Train', X_train, y_train), ('Test', X_test, y_test)]:
    preds = clf.predict(x_eval)
    mae = mean_absolute_error(y_eval, preds)
    rmse = np.sqrt(mean_squared_error(y_eval, preds))
    r2 = r2_score(y_eval, preds)
    print(f'XGBoost {name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}')

#### Optimizing XGBOOST, SVM, RF, DT and KNN with Best Hyperparameters

In [ ]:
# Training all models with their best hyperparameters found during tuning

# 1. XGBOOST+
model_xgb = xbt.XGBRegressor(learning_rate=0.2, max_depth=4, n_estimators=1000, random_state=42)
model_xgb.fit(X_train, y_train)

# 2. SVM (Uses scaled data)
model_svm_final = svm.SVR(C=10, gamma='auto', kernel='rbf')
model_svm_final.fit(Xs_train, ys_train)

# 3. RANDOM FOREST
model_rf_final = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42)
model_rf_final.fit(X_train, y_train)

# 4. DECISION TREE
model_dt_final = DecisionTreeRegressor(max_depth=10, min_samples_split=2, random_state=42)
model_dt_final.fit(X_train, y_train)

# 6. KNN (Uses scaled data)
model_knn_final = KNeighborsRegressor(n_neighbors=5, weights='distance')
model_knn_final.fit(Xs_train, ys_train)


print("All models have been successfully trained with optimized parameters.")

#### Training MLR, Material and Printing Parameter Models

In [ ]:
# 5. MULTILINEAR REGRESSION
model_mlr_final = LinearRegression()
model_mlr_final.fit(X_train, y_train)

# MATERIAL AND PROCESS PARAMETER MODELS
# 7. Material Model (XGBOOST)
model_mat_final = xbt.XGBRegressor(learning_rate=0.2, max_depth=4, n_estimators=1000,random_state=42)
model_mat_final.fit(X_train_mat, y_train_mat)

# 8. Printing Model (XGBOOST)
model_print_final = xbt.XGBRegressor(learning_rate=0.2, max_depth=4, n_estimators=1000,random_state=42)
model_print_final.fit(X_train_print, y_train_print)

### Evaluation of Models

In [ ]:
evaluation_config = [
    ('XGBOOST+', model_xgb, X_train, y_train, X_test, y_test, False),
    ('SVM', model_svm_final, Xs_train, ys_train, Xs_test, ys_test, True),
    ('RandomForest', model_rf_final, X_train, y_train, X_test, y_test, False),
    ('DecisionTree', model_dt_final, X_train, y_train, X_test, y_test, False),
    ('MLR', model_mlr_final, X_train, y_train, X_test, y_test, False),
    ('KNN', model_knn_final, Xs_train, ys_train, Xs_test, ys_test, True),
    ('Material Model', model_mat_final, X_train_mat, y_train_mat, X_test_mat, y_test_mat, False),
    ('Printing Model', model_print_final, X_train_print, y_train_print, X_test_print, y_test_print, False)
]

print(f"{'Model Name':<15} | {'Set':<6} | {'MAE':<8} | {'RMSE':<8} | {'R2':<8}")
print("-" * 55)

for name, model, xtr, ytr, xts, yts, scaled in evaluation_config:
    for d_name, x_eval, y_eval in [('Train', xtr, ytr), ('Test', xts, yts)]:
        preds = model.predict(x_eval)

        if scaled:
            y_act = y_eval * df_range[target_col] + df_min[target_col]
            p_act = preds * df_range[target_col] + df_min[target_col]
        else:
            y_act, p_act = y_eval, preds

        mae = mean_absolute_error(y_act, p_act)
        rmse = np.sqrt(mean_squared_error(y_act, p_act))
        r2 = r2_score(y_act, p_act)

        print(f"{name:<15} | {d_name:<6} | {mae:<8.4f} | {rmse:<8.4f} | {r2:<8.4f}")

## VISUALIZATION OF MODEL PERFORMANCE

### Learning Curves

In [ ]:
def plot_learning_curve_mse(estimator, title, X_data, y_data, axes, ylim=None, cv=10, n_jobs=-1, train_sizes=np.linspace(.1, 1.0, 10), is_scaled=False):
    # Using neg_mean_squared_error for MSE
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X_data, y_data, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring='neg_mean_squared_error')

    # Convert negative MSE to positive
    train_scores_mean = (-train_scores).mean(axis=1)
    train_scores_std = (-train_scores).std(axis=1)
    test_scores_mean = (-test_scores).mean(axis=1)
    test_scores_std = (-test_scores).std(axis=1)

    # Inverse scale the MSE if the model was trained on scaled data
    if is_scaled:
        # Since MSE is in units^2, we multiply by the square of the range
        scale_factor = (df_range[target_col]) ** 2
        train_scores_mean *= scale_factor
        train_scores_std *= scale_factor
        test_scores_mean *= scale_factor
        test_scores_std *= scale_factor

    axes.grid(True, linestyle='--', alpha=0.7)
    axes.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r", label="Training std")
    axes.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color="g", label="CV std")

    axes.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    axes.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")

    axes.xaxis.set_major_locator(ticker.MultipleLocator(25))
    axes.set_title(title, fontsize=10, fontweight='bold')
    axes.set_xlabel("Training examples", fontsize=8)
    axes.set_ylabel("MSE (MPa$^2$)", fontsize=8)
    if ylim:
        axes.set_ylim(*ylim)
    axes.legend(loc="best", fontsize=7)

# Define models to plot using the optimized_models dictionary
models_mse = [
    (optimized_models['XGBoost'], "XGBOOST+", X_train, y_train, False),
    (optimized_models['SVM'], "SVM", Xs_train, ys_train, True),
    (optimized_models['RandomForest'], "RF", X_train, y_train, False),
    (optimized_models['DecisionTree'], "DT", X_train, y_train, False),
    (model_mlr_final, "MLR", X_train, y_train, False),
    (optimized_models['KNN'], "KNN", Xs_train, ys_train, True)
]

# Set a common Y-limit for all MSE plots for easier comparison
common_ylim_mse = (-200, 2000)

for model_obj, model_name, x_curr, y_curr, is_scaled in models_mse:
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_learning_curve_mse(model_obj, f"Learning Curve (MSE): {model_name}", x_curr, y_curr, ax, ylim=common_ylim_mse, is_scaled=is_scaled)

    filename = f"LC_MSE_{model_name.replace(' ', '_')}.jpg"
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    print(f"Saved: {filename}")
    plt.close(fig)

### XGBOOST+ Error Loss Curves

In [ ]:
# To get loss over epochs, we use the final optimized XGBoost model
# We re-fit with an eval_set to capture the progression
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

results = model_xgb.evals_result()
epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

# Plotting MSE (Internal XGBoost RMSE squared)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_axis, np.square(results['validation_0']['rmse']), label='Train Loss (MSE)')
ax.plot(x_axis, np.square(results['validation_1']['rmse']), label='Validation Loss (MSE)')
ax.legend()
plt.ylabel('MSE (MPa$^2$)')
plt.xlabel('Epochs (Boosting Rounds)')
plt.title('XGBOOST+ Training Loss Over Epochs')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('XGBoost_Loss_Epochs.jpg', dpi=300)
plt.close()

### Scatter Plot of all Models

In [ ]:
def plot_model_scatter(name, y_true, y_pred, r2_val, filename):
    deviation = 0.10 * y_true
    lower_bound = y_true - deviation
    upper_bound = y_true + deviation

    plt.figure(figsize=(3.2, 3.1))
    plt.scatter(y_true, y_pred, color='green', alpha=0.5, label='Prediction')
    plt.scatter(y_true, y_true, color='blue', alpha=0.5, label='Actual')
    plt.plot(y_true, lower_bound, color='black', linewidth=0.5, label='10% Deviation')
    plt.plot(y_true, upper_bound, color='black', linewidth=0.5)

    m, b = np.polyfit(y_true, y_pred, 1)
    plt.plot(y_true, m * y_true + b, 'r--', lw=1)

    plt.xlabel('Actual Compressive Strength (MPa)', fontsize=7)
    plt.ylabel('Predicted Compressive Strength (MPa)', fontsize=7)
    plt.title(f'{name} (R² = {r2_val:.3f})', fontsize=7)
    plt.xlim(0, 175)
    plt.xticks(np.arange(0, 200, 25))
    plt.grid(True)
    plt.legend(fontsize=7, shadow=True, facecolor='white')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

# Helper to get R2 and Preds for current models
def get_stats(model, x, y, is_scaled=False):
    p = model.predict(x)
    if is_scaled:
        y_act = y * df_range[target_col] + df_min[target_col]
        p_act = p * df_range[target_col] + df_min[target_col]
    else:
        y_act, p_act = y, p
    return p_act, r2_score(y_act, p_act)

# Mapping plots to currently defined final models
plots_config = [
    ('XGBOOST+', *get_stats(model_xgb, X_test, y_test), 'R2_XGBOOST_plus.jpg'),
    ('SVM', *get_stats(model_svm_final, Xs_test, ys_test, True), 'R2_SVM.jpg'),
    ('RF', *get_stats(model_rf_final, X_test, y_test), 'R2_RF.jpg'),
    ('DT', *get_stats(model_dt_final, X_test, y_test), 'R2_DT.jpg'),
    ('MLR', *get_stats(model_mlr_final, X_test, y_test), 'R2_MLR.jpg'),
    ('KNN', *get_stats(model_knn_final, Xs_test, ys_test, True), 'R2_KNN.jpg'),
    ('Material Model', *get_stats(model_mat_final, X_test_mat, y_test_mat), 'R2_Material.jpg'),
    ('Printing Model', *get_stats(model_print_final, X_test_print, y_test_print), 'R2_Printing.jpg')
]

for name, y_p, r2_v, fname in plots_config:
    # Determine correct ground truth (y_test) based on model name
    if 'Material' in name:
        y_true_set = y_test_mat
    elif 'Printing' in name:
        y_true_set = y_test_print
    else:
        y_true_set = y_test

    plot_model_scatter(name, y_true_set, y_p, r2_v, fname)

### Residual Plots

In [ ]:
def plot_residuals_individual(name, y_true, y_pred, ylim=None):
    plt.figure(figsize=(8, 6), facecolor='white')
    residuals = y_true - y_pred
    plt.scatter(y_pred, residuals, alpha=0.8, color='purple', edgecolors='white', s=80, linewidth=0.5)
    plt.axhline(0, color='red', linestyle='-', lw=2.5, alpha=1.0)
    plt.title(f'Residual Plot: {name}', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Compressive Strength (MPa)', fontsize=12)
    plt.ylabel('Residuals', fontsize=12)
    if ylim:
        plt.ylim(ylim)
    plt.grid(True, linestyle=':', alpha=0.5, color='gray')
    filename = f"Residuals_{name.replace('+', 'plus').replace(' ', '_')}.jpg"
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    print(f"Saved: {filename}")
    plt.close()

# Helper to get unscaled predictions for residual plots
def get_preds_unscaled(model, x, y, is_scaled=False):
    p = model.predict(x)
    if is_scaled:
        p_act = p * df_range[target_col] + df_min[target_col]
    else:
        p_act = p
    return p_act

model_results = [
    ('XGBOOST+', get_preds_unscaled(model_xgb, X_test, y_test)),
    ('SVM', get_preds_unscaled(model_svm_final, Xs_test, ys_test, True)),
    ('RF', get_preds_unscaled(model_rf_final, X_test, y_test)),
    ('DT', get_preds_unscaled(model_dt_final, X_test, y_test)),
    ('MLR', get_preds_unscaled(model_mlr_final, X_test, y_test)),
    ('KNN', get_preds_unscaled(model_knn_final, Xs_test, ys_test, True))
]

common_resid_ylim = (-40, 40)

for name, y_p in model_results:
    plot_residuals_individual(name, y_test, y_p, ylim=common_resid_ylim)

### SHAP ANALYSIS

In [ ]:
# SHAP values using the final XGBoost model
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Plot SHAP summary with minimized height
plt.figure(figsize=(4, 4))
shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
plt.gca().set_title(" ")
plt.savefig("SHAP1.jpg")
plt.close()

In [ ]:
# Waterfall Plot using the final XGBoost model
exp = shap.Explainer(model_xgb)
shap_values_exp = exp(X_test)
plt.figure(figsize=(3.2, 3.4))
shap.plots.waterfall(shap_values_exp[0], show=False)
plt.tight_layout()
plt.savefig("WATERFALL.jpg")
plt.close()

In [ ]:
#Feature Importance using the Explanation object
plt.figure(figsize=(4, 2))
# We use the already computed shap_values_exp which is an Explanation object
shap.plots.bar(shap_values_exp, show=False)
plt.tight_layout()
plt.savefig("BAR_IMPORTANCE.jpg")
plt.close()

In [ ]:
# Partial Dependence Plot (PDP) using the final XGBoost model
for i in X_test.columns:
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    PartialDependenceDisplay.from_estimator(model_xgb, X_test, [i], ax=ax)
    ax.set_xlabel(f"{i}", fontsize=8)
    ax.set_ylabel("Partial Dependence", fontsize=8)
    ax.set_title(" ")
    plt.tight_layout()
    plt.savefig(f" PDP of {i.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')}.jpg")
    plt.close()

# UPLOAD EXCEL OR CSV FILE TO PREDICT COMPRESSIVE STRENGTH USING THE XGBOOST+ MODEL

In [ ]:
print("Please upload your file (.xlsx or .csv) containing the feature columns:")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    file_content = io.BytesIO(uploaded[file_name])

    if file_name.endswith('.csv'):
        user_df = pd.read_csv(file_content, encoding='latin-1')
    else:
        user_df = pd.read_excel(file_content)

    required_features = X.columns.tolist()

    priority_features = ['Cement (Kg/m³)', 'Water (Kg/m³)', 'Age (Days)', 'Nozzle Speed (mm/s)',
                         'Fine Aggregate (Kg/m³)', 'Silica Fume (Kg/m³)', 'Fly ash (Kg/m³)',
                         'DIA (mm)', 'Layer Depth (mm)', 'Layer Width (mm)']

    other_features = [f for f in required_features if f not in priority_features]
    mapping_order = sorted(priority_features, key=len, reverse=True) + sorted(other_features, key=len, reverse=True)

    model_input = pd.DataFrame(index=user_df.index)
    user_cols = [str(c).strip() for c in user_df.columns]
    user_cols_lower = [c.lower() for c in user_cols]
    used_indices = set()

    for feature in mapping_order:
        found = False
        feature_keyword = feature.lower().split('(')[0].strip()

        # Step A: Exact Match
        for i, u_col in enumerate(user_cols_lower):
            if i not in used_indices and (feature_keyword == u_col or feature.lower() == u_col):
                model_input[feature] = user_df.iloc[:, i]
                used_indices.add(i)
                found = True
                break

        # Step B: Keyword Match
        if not found:
            for i, u_col in enumerate(user_cols_lower):
                if i not in used_indices and feature_keyword in u_col:
                    model_input[feature] = user_df.iloc[:, i]
                    used_indices.add(i)
                    found = True
                    break

        # Step C: Reverse Keyword Match
        if not found:
            for i, u_col in enumerate(user_cols_lower):
                clean_u_col = u_col.split('(')[0].strip()
                if i not in used_indices and len(clean_u_col) > 2 and clean_u_col in feature_keyword:
                    model_input[feature] = user_df.iloc[:, i]
                    used_indices.add(i)
                    found = True
                    break

        if not found:
            model_input[feature] = 0

    model_input = model_input[required_features].fillna(0)
    predictions = model_xgb.predict(model_input)
    user_df['PREDICTED_Compressive_Strength_MPa'] = predictions

    output_file = 'Predictions_Output.xlsx'
    user_df.to_excel(output_file, index=False)
    print(f"\nProcessing Complete! Downloading {output_file}...")
    files.download(output_file)
else:
    print('No file uploaded.')